# PathSnap: Exploratory Data Analysis (Phase 1)

This notebook reproduces the project EDA from the repository. It analyses the **train** and **student_test** partitions in the processed summary tables. Raw timestamp-gap and speed plots use **training trips only**. The evaluator partition and its labels must not be used.

**Run from top to bottom in Google Colab.** The setup cells mount Drive, locate `Student folder.zip`, prepare the repository, and build the processed tables. If your ZIP is stored somewhere else, change `ZIP_PATH` in the dataset setup cell.

## 0. Environment and repository setup

This cell reuses `/content/PT-007-PathSnap` if it already exists. It clones only when the folder is absent, so it will not attempt to clone a repository into itself.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import zipfile

REPO = Path("/content/PT-007-PathSnap")
REPO_URL = "https://github.com/Amara1404/PT-007-PathSnap.git"

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    print(f"Using existing repository: {REPO}")

# If GitHub does not yet contain the Phase 1 source files, upload the
# phase1 ZIP you downloaded from your Mac and overlay its project files.
eda_module = REPO / "src" / "viz" / "eda.py"
if not eda_module.exists():
    from google.colab import files
    print("The cloned repo is missing src/viz/eda.py.")
    print("Upload PT-007-PathSnap_phase1.zip from your computer when prompted.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise FileNotFoundError("No ZIP was uploaded. Upload PT-007-PathSnap_phase1.zip.")
    phase1_zip = Path("/content") / zip_names[0]
    phase1_extract = Path("/content/pathsnap_phase1_extract")
    if phase1_extract.exists():
        shutil.rmtree(phase1_extract)
    phase1_extract.mkdir(parents=True)
    with zipfile.ZipFile(phase1_zip, "r") as zf:
        zf.extractall(phase1_extract)
    roots = [p for p in phase1_extract.rglob("eda.py") if p.as_posix().endswith("src/viz/eda.py")]
    if not roots:
        raise FileNotFoundError("Uploaded ZIP does not contain src/viz/eda.py.")
    project_root = roots[0].parents[2]
    shutil.copytree(project_root, REPO, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".git"))
    print("Phase 1 source files copied into the Colab repository.")

sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("Repository:", REPO)
print("EDA module:", (REPO / "src" / "viz" / "eda.py").exists())
print("Working directory:", Path.cwd())

## 1. Mount Drive and locate the raw dataset

The ZIP must contain a `Student folder/` directory with `train/` and `student_test/` subdirectories (or those two directories at its root). If Drive cannot find it automatically, set `ZIP_PATH` to the exact path shown in the Colab Files panel.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Preferred location based on the project setup. Edit this if your ZIP is elsewhere.
ZIP_PATH = Path("/content/drive/MyDrive/Student folder.zip")

if not ZIP_PATH.exists():
    possible = list(Path("/content/drive/MyDrive").glob("**/Student folder.zip"))
    if possible:
        ZIP_PATH = possible[0]

print("Dataset ZIP:", ZIP_PATH)
if not ZIP_PATH.exists():
    raise FileNotFoundError(
        "Could not find Student folder.zip in My Drive. Copy it from Shared with me "
        "into My Drive, or edit ZIP_PATH to its exact location."
    )

## 2. Extract and verify the dataset

Extraction is done into a temporary folder first. The notebook then identifies the directory containing `train/` and `student_test/` and copies it to the repository root as `Student folder/`.

In [ ]:
import shutil
import zipfile

extract_root = Path("/content/pathsnap_dataset_extract")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(extract_root)

# Find the directory that contains the partition folders.
train_dirs = [p for p in extract_root.rglob("train") if p.is_dir()]
source_root = None
for train_dir in train_dirs:
    if (train_dir.parent / "student_test").is_dir():
        source_root = train_dir.parent
        break
if source_root is None and train_dirs:
    source_root = train_dirs[0].parent

if source_root is None:
    raise FileNotFoundError("The ZIP does not contain a train/ directory.")

DATA_ROOT = REPO / "Student folder"
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
shutil.copytree(source_root, DATA_ROOT)

TRAIN_RAW = DATA_ROOT / "train"
STUDENT_TEST_RAW = DATA_ROOT / "student_test"
print("Dataset root:", DATA_ROOT)
print("Training directory exists:", TRAIN_RAW.is_dir())
print("Student-test directory exists:", STUDENT_TEST_RAW.is_dir())
print("Training trip folders:", sum(p.is_dir() for p in TRAIN_RAW.iterdir()) if TRAIN_RAW.is_dir() else 0)
print("Student-test trip folders:", sum(p.is_dir() for p in STUDENT_TEST_RAW.iterdir()) if STUDENT_TEST_RAW.is_dir() else 0)
assert TRAIN_RAW.is_dir(), "Missing Student folder/train"
assert STUDENT_TEST_RAW.is_dir(), "Missing Student folder/student_test"

## 3. Install required packages and build processed datasets

The build step creates the inventory, network statistics, data-quality summary, candidate summary and parquet datasets. It may take a few minutes. Do not interrupt it while it is processing trips.

In [ ]:
%pip -q install pyarrow folium

# Build all partitions from the supplied raw data.
subprocess.run(
    [sys.executable, "-m", "src.data.build_datasets",
     "--raw", str(DATA_ROOT), "--out", str(REPO / "data" / "processed")],
    cwd=REPO, check=True
)

PROC = REPO / "data" / "processed"
FIG = REPO / "results" / "figures"
MAPS = REPO / "results" / "maps"
FIG.mkdir(parents=True, exist_ok=True)
MAPS.mkdir(parents=True, exist_ok=True)

required = ["inventory_trips.csv", "network_stats.csv", "data_quality_issues.csv", "candidate_summary.csv"]
for name in required:
    path = PROC / name
    print(name, "exists:", path.exists())
    assert path.exists(), f"Dataset build did not produce {name}"

## 4. Load EDA tables and inspect the dataset

`inventory_trips.csv` contains per-trip counts, sampling intervals, geographic extent and reference-route alignment. `network_stats.csv` describes each trip road network. `data_quality_issues.csv` records issues found during preprocessing.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import Image, display, HTML
from src.viz import eda

inv, net, iss, both = eda.load_tables(PROC)

print("Inventory shape:", inv.shape)
print("Network stats shape:", net.shape)
print("Issues shape:", iss.shape)
print("Partitions:")
display(inv.partition.value_counts().rename_axis("partition").to_frame("trips"))
print("Candidate summary:")
display(pd.read_csv(PROC / "candidate_summary.csv"))

assert set(inv.partition.unique()).issubset({"train", "student_test", "eval"})
print("No evaluator labels are loaded by this notebook.")

## 5. Trips per partition and points per trip

In [ ]:
display(inv.groupby("partition").agg(
    trips=("trip_id", "nunique"),
    total_points=("n_obs", "sum"),
    median_points=("n_obs", "median"),
    min_points=("n_obs", "min"),
    max_points=("n_obs", "max")
).round(1))
display(Image(filename=str(eda.fig_inventory(inv, FIG))))

**Interpretation prompt:** Compare the number of trips and points in each partition. Comment on the spread of trip lengths and whether a small number of trips are much longer than the others. Use the table above rather than guessing.

## 6. Time gaps between consecutive GPS observations

Raw-step analysis is restricted to the training partition.

In [ ]:
steps = eda.raw_dt_speed(TRAIN_RAW)
print("Training raw trips analysed:", steps.trip_id.nunique())
print("Training GPS steps:", len(steps))
print("Share of steps exactly 1 second apart:", round((steps.dt == 1).mean() * 100, 2), "%")
print("Steps with gaps > 5 seconds:", int((steps.dt > 5).sum()))
display(Image(filename=str(eda.fig_time_gaps(inv, steps, FIG))))
display(inv.sort_values("dt_max_s", ascending=False)[
    ["trip_id", "partition", "n_obs", "dt_max_s", "n_gaps_gt_5s"]
].head(10))

**Interpretation prompt:** Explain the typical sampling interval, how often gaps above five seconds occur, and why very large timestamp jumps make duration and average-speed calculations misleading.

## 7. Speed outliers

Speed is calculated from the distance between consecutive GPS points divided by the elapsed time. The 250 km/h line is the project threshold used to flag implausible steps.

In [ ]:
speed_stats = steps.groupby("trip_id").agg(
    n_steps=("speed_kmh", "size"),
    max_speed_kmh=("speed_kmh", "max"),
    n_speed_gt_250=("speed_kmh", lambda s: int((s > 250).sum()))
).sort_values("max_speed_kmh", ascending=False)

display(Image(filename=str(eda.fig_speed(inv, steps, FIG))))
display(speed_stats.head(15))
print("Total training steps above 250 km/h:", int((steps.speed_kmh > 250).sum()))

**Interpretation prompt:** Identify whether high-speed steps are spread evenly or concentrated in a few trips. Explain why isolated GPS jumps can create unrealistic calculated speeds.

## 8. Road-network size and connectivity

In [ ]:
network_columns = [
    "n_nodes", "n_arcs", "n_weak_components",
    "largest_weak_component_frac", "n_isolated_nodes", "two_way_arc_frac"
]
display(both.groupby("partition")[network_columns].median().round(3))
display(Image(filename=str(eda.fig_network(both, FIG))))
display(both[network_columns].describe().round(3))

**Interpretation prompt:** Discuss the range of network sizes and the number of weakly connected components. Explain why disconnected components matter when a later HMM tries to make transitions between candidate road arcs.

## 9. Data-quality issues and reference-route quality

In [ ]:
if not iss.empty:
    display(iss.groupby("issue").agg(
        trips=("trip_id", "nunique"),
        total=("count", "sum")
    ).sort_values("trips", ascending=False))
else:
    print("No data-quality issues were recorded.")

display(Image(filename=str(eda.fig_issues(iss, FIG))))
display(Image(filename=str(eda.fig_label_quality(inv, FIG))))

display(inv.sort_values("frac_label_low_conf", ascending=False)[
    ["trip_id", "partition", "frac_label_low_conf", "label_dist_median_m", "label_dist_p95_m"]
].head(12))

**Interpretation prompt:** Explain what it means when GPS points are far from the reference route. Discuss why trips 00000033 and 00000054 were flagged as having unreliable labels, and why unreliable labels should not be treated as correct training targets.

## 10. Geographic extent

In [ ]:
display(Image(filename=str(eda.fig_extent(inv, FIG))))
display(inv.groupby("partition")[
    ["extent_x_km", "extent_y_km"]
].median().round(2))

**Interpretation prompt:** Describe the geographic spread of the trips and explain why the project uses a local metric projection for distance and speed calculations rather than calculating directly in longitude/latitude degrees.

## 11. Case studies: trips 33, 54 and 10

In [ ]:
for trip_id in ["00000033", "00000054", "00000010"]:
    trip_dir = TRAIN_RAW / trip_id
    if not trip_dir.is_dir():
        print(f"Trip {trip_id} not found; skipping")
        continue
    print(f"Trip {trip_id}")
    display(Image(filename=str(eda.fig_trip_overview(trip_dir, FIG))))

**Interpretation prompt:** Compare trips 33 and 54 with trip 10. Describe the visual difference between GPS observations and the reference route. State what the visual evidence can and cannot establish about why the reference route differs from the GPS trace.

## 12. Generate interactive Folium maps

In [ ]:
from src.viz.trip_layers import load_layers
import inspect

print("load_layers signature:")
print(inspect.signature(load_layers))

print("\nload_layers implementation:")
print(inspect.getsource(load_layers))

In [ ]:
import inspect
from src.viz.maps import trip_map

print(inspect.signature(trip_map))

In [ ]:
file_path = "/content/PT-007-PathSnap/src/viz/maps.py"

with open(file_path, "r") as f:
    code = f.read()

print(code[:2000])

In [ ]:
file_path = "/content/PT-007-PathSnap/src/viz/maps.py"

with open(file_path, "r") as f:
    code = f.read()

code = code.replace(
    'import folium',
    'import folium\nimport numpy as np'
)

code = code.replace(
    'pts = list(gps or [])\n    seg_pts = [p for s in (predicted or []) + (reference or []) for p in s]\n    allp = pts + seg_pts',
    '''pts = [] if gps is None else np.asarray(gps).tolist()
    predicted = [] if predicted is None else list(predicted)
    reference = [] if reference is None else list(reference)
    network = [] if network is None else list(network)
    extra = [] if extra is None else list(extra)

    seg_pts = [p for s in predicted + reference for p in s]
    allp = pts + seg_pts'''
)

with open(file_path, "w") as f:
    f.write(code)

print("maps.py updated!")

In [ ]:
import importlib
import src.viz.maps

importlib.reload(src.viz.maps)

from src.viz.maps import trip_map

print("Map module reloaded successfully!")

In [ ]:
from src.viz.trip_layers import load_layers
from src.viz.maps import trip_map
from IPython.display import display

for trip_id in ["00000033", "00000054", "00000010"]:

    trip_dir = TRAIN_RAW / trip_id

    if not trip_dir.is_dir():
        print(f"Trip {trip_id} not found. Skipping.")
        continue

    # Load GPS, network and reference route layers
    layers = load_layers(trip_dir)

    # Generate map
    fmap = trip_map(
        gps=layers["gps"],
        network=layers["network"],
        reference=layers["reference"],
        zoom_start=15
    )

    # Save map
    out_path = MAPS / f"eda_trip_{trip_id}.html"
    fmap.save(str(out_path))

    print(f"Successfully saved: {out_path}")

    display(fmap)

## 13. Save and verify EDA outputs

In [ ]:
# Save raw per-step statistics so the report can cite reproducible outputs.
raw_stats = steps.groupby("trip_id").agg(
    n_steps=("dt", "size"),
    dt_max_s=("dt", "max"),
    speed_max_kmh=("speed_kmh", "max"),
    n_speed_gt_250=("speed_kmh", lambda s: int((s > 250).sum()))
).reset_index()
raw_stats.to_csv(REPO / "results" / "eda_raw_step_stats.csv", index=False)

print("Figures:")
for p in sorted(FIG.glob("*.png")):
    print(p.relative_to(REPO), f"({p.stat().st_size:,} bytes)")
print("Interactive maps:")
for p in sorted(MAPS.glob("*.html")):
    print(p.relative_to(REPO), f"({p.stat().st_size:,} bytes)")
print("Raw statistics:", (REPO / "results" / "eda_raw_step_stats.csv").exists())

In [ ]:
from pathlib import Path

print("MAPS:")
for f in MAPS.glob("*"):
    print(f)

print("\nEDA OUTPUTS:")
for f in Path("/content/PT-007-PathSnap").rglob("*"):
    if f.is_file() and f.suffix in [".png", ".csv", ".html", ".json"]:
        print(f)

In [ ]:
import shutil

PROJECT = Path("/content/PT-007-PathSnap")

shutil.make_archive(
    "/content/PathSnap_EDA_Results",
    "zip",
    root_dir=PROJECT,
    base_dir="results"
)

print("EDA results ZIP created successfully!")

In [ ]:
from google.colab import files

files.download("/content/PathSnap_EDA_Results.zip")